In [2]:
!wget https://zenodo.org/record/7387261/files/dwug_en.zip -O dwug_en.zip
!unzip dwug_en.zip -d DWUG-English

--2026-09-25 20:14:13--  https://zenodo.org/record/7387261/files/dwug_en.zip
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.103.118, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/7387261/files/dwug_en.zip [following]
--2026-09-25 20:14:13--  https://zenodo.org/records/7387261/files/dwug_en.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 11468345 (11M) [application/octet-stream]
Saving to: ‘dwug_en.zip’

dwug_en.zip         100%[===================>]  10.94M  3.16MB/s    in 4.9s    

2026-09-25 20:14:18 (2.23 MB/s) - ‘dwug_en.zip’ saved [11468345/11468345]

Archive:  dwug_en.zip
   creating: DWUG-English/dwug_en/
   creating: DWUG-English/dwug_en/stats/
   creating: DWUG-English/dwug_en/stats/opt/
  inflating: DWUG-English/dwug_en/stats/opt/stats.csv  
  inflating: DWUG-English/dwug_en/stats

In [3]:
!ls DWUG-English/dwug_en

annotators.csv	clusters  data	graphs	guidelines  plots  README.html	stats


In [4]:
!ls DWUG-English/dwug_en/data | head -20

afternoon_nn
attack_nn
bag_nn
ball_nn
bar_nn
bit_nn
chairman_nn
chef_nn
circle_vb
contemplation_nn
donkey_nn
edge_nn
face_nn
fiction_nn
gas_nn
graft_nn
grain_nn
head_nn
heel_nn
include_vb


In [5]:
!ls DWUG-English/dwug_en/data/afternoon_nn

judgments.csv  uses.csv


In [6]:
import pandas as pd

uses = pd.read_csv("DWUG-English/dwug_en/data/afternoon_nn/uses.csv", sep="\t")
print(uses.shape)
print(uses.columns.tolist())
print(uses.head(3))

(200, 14)
['lemma', 'pos', 'date', 'grouping', 'identifier', 'description', 'context', 'indexes_target_token', 'indexes_target_sentence', 'context_tokenized', 'indexes_target_token_tokenized', 'indexes_target_sentence_tokenized', 'context_lemmatized', 'context_pos']
          lemma   pos  date  grouping                 identifier description  \
0  afternoon_nn  nnt1  1857         1  nf_1857_753133.txt-1759-5               
1  afternoon_nn  nnt1  1855         1  fic_1855_2965.txt-1490-13               
2  afternoon_nn  nnt1  1839         1   fic_1839_7352.txt-2123-7               

                                             context indexes_target_token  \
0  One little incident in our afternoon travel I ...                27:36   
1  Substituting it for the gold ring, he wore it ...                61:70   
2  On one occasion they started on their afternoo...                38:47   

  indexes_target_sentence                                  context_tokenized  \
0                   0:1

In [7]:
judgments = pd.read_csv("DWUG-English/dwug_en/data/afternoon_nn/judgments.csv", sep="\t")
print(judgments.shape)
print(judgments.columns.tolist())
print(judgments.head(3))

(427, 7)
['identifier1', 'identifier2', 'annotator', 'judgment', 'comment', 'lemma', 'round']
                  identifier1                identifier2   annotator  \
0  mag_1986_492553.txt-134-22  fic_1992_40272.txt-627-18  annotator2   
1     fic_1831_7210.txt-510-4  fic_1846_7036.txt-2381-16  annotator2   
2   mag_1850_565942.txt-396-5    fic_1858_9768.txt-412-5  annotator2   

   judgment comment         lemma  round  
0       4.0          afternoon_nn      2  
1       4.0          afternoon_nn      2  
2       4.0          afternoon_nn      2  


In [8]:
uses = pd.read_csv("DWUG-English/dwug_en/data/afternoon_nn/uses.csv", sep="\t")
judgments = pd.read_csv("DWUG-English/dwug_en/data/afternoon_nn/judgments.csv", sep="\t")

# lookup: identifier -> sentence text
id_to_context = uses.set_index("identifier")["context"]

# attach sentence text for both sides of each judgment
judgments["sentence1"] = judgments["identifier1"].map(id_to_context)
judgments["sentence2"] = judgments["identifier2"].map(id_to_context)

print(judgments[["sentence1", "sentence2", "judgment"]].head(5))

                                           sentence1  \
0   Well,  Mike Maloney said, " it's because whil...   
1  He had passed the afternoon in his favourite r...   
2  rrhe next day in the afternoon, " he exercised...   
3  Shortly after, an officer looked into his cell...   
4  Ari arrived at Kibbutz Revivim Tuesday afterno...   

                                           sentence2  judgment  
0  Old shopping lists and ticket stubs and wads o...       4.0  
1  By some accident, Big Elk became separated fro...       4.0  
2  About four o'clock in the afternoon, after ref...       4.0  
3  On those afternoons by the sea, we knitted, Sh...       4.0  
4  Old shopping lists and ticket stubs and wads o...       3.0  


In [9]:
pair_gold = (
    judgments
    .groupby(["identifier1", "identifier2"])
    .agg(
        sentence1=("sentence1", "first"),
        sentence2=("sentence2", "first"),
        mean_judgment=("judgment", "mean"),
        n_annotators=("judgment", "count")
    )
    .reset_index()
)

print(pair_gold.head())
print(pair_gold.shape)

                 identifier1                identifier2  \
0   fic_1822_7275.txt-216-23   fic_1850_2310.txt-1321-9   
1   fic_1822_7275.txt-530-41   fic_1971_14051.txt-667-2   
2   fic_1825_7041.txt-1647-2   fic_1850_2310.txt-1321-9   
3  fic_1827_9758.txt-5747-16  fic_1846_7036.txt-2381-16   
4    fic_1831_7210.txt-510-4  fic_1846_7036.txt-2381-16   

                                           sentence1  \
0  Mrs. Daggett, nettled by her sister's hint, ro...   
1  Elvira had just got possession, by stealth, of...   
2  On the afternoon of the same day that Wilson c...   
3  Being obliged to leave Solebury the next day, ...   
4  He had passed the afternoon in his favourite r...   

                                           sentence2  mean_judgment  \
0  She remained in company with the cows all the ...       4.000000   
1  On those afternoons by the sea, we knitted, Sh...       3.666667   
2  She remained in company with the cows all the ...       3.500000   
3  By some accident, Big

In [10]:
import os

words = os.listdir("DWUG-English/dwug_en/data")[:15]  # pick your sample size here
all_pairs = []

for word in words:
    try:
        u = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/uses.csv", sep="\t")
        j = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/judgments.csv", sep="\t")
        id2ctx = u.set_index("identifier")["context"]
        j["sentence1"] = j["identifier1"].map(id2ctx)
        j["sentence2"] = j["identifier2"].map(id2ctx)
        pg = (j.groupby(["identifier1","identifier2"])
                .agg(sentence1=("sentence1","first"),
                     sentence2=("sentence2","first"),
                     mean_judgment=("judgment","mean"),
                     n_annotators=("judgment","count"))
                .reset_index())
        pg["lemma"] = word
        all_pairs.append(pg)
    except Exception as e:
        print(f"skipped {word}: {e}")

full_dataset = pd.concat(all_pairs, ignore_index=True)
print(full_dataset.shape)
print(full_dataset.head())

(10359, 7)
                identifier1                 identifier2  \
0  fic_1812_8652.txt-166-10    fic_1821_8837.txt-339-72   
1  fic_1812_8652.txt-166-10   fic_1960_4443.txt-1803-14   
2  fic_1812_8652.txt-166-10  fic_1969_781999.txt-7587-7   
3  fic_1812_8652.txt-166-10    fic_1972_23885.txt-275-0   
4  fic_1812_8652.txt-166-10     fic_2006_27121.txt-1-27   

                                           sentence1  \
0  Gives me my son such as he was... the prop, Th...   
1  Gives me my son such as he was... the prop, Th...   
2  Gives me my son such as he was... the prop, Th...   
3  Gives me my son such as he was... the prop, Th...   
4  Gives me my son such as he was... the prop, Th...   

                                           sentence2  mean_judgment  \
0  What most of all surprised me in this man was,...       4.000000   
1  ... eating at until I was an empty shell, unti...       3.000000   
2  Almost lost in the shadows were domestic props...       1.500000   
3  props, sti

In [11]:
print(judgments["judgment"].unique())

[4. 3. 0. 2.]


In [12]:
import os
import pandas as pd

words = os.listdir("DWUG-English/dwug_en/data")[:15]
all_pairs = []

for word in words:
    try:
        u = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/uses.csv", sep="\t")
        j = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/judgments.csv", sep="\t")
        j = j[j["judgment"] >= 1]
        id2ctx = u.set_index("identifier")["context"]
        j["sentence1"] = j["identifier1"].map(id2ctx)
        j["sentence2"] = j["identifier2"].map(id2ctx)
        pg = (j.groupby(["identifier1","identifier2"])
                .agg(sentence1=("sentence1","first"),
                     sentence2=("sentence2","first"),
                     mean_judgment=("judgment","mean"),
                     n_annotators=("judgment","count"))
                .reset_index())
        pg["lemma"] = word
        all_pairs.append(pg)
    except Exception as e:
        print(f"skipped {word}: {e}")

full_dataset = pd.concat(all_pairs, ignore_index=True)
print(full_dataset.shape)
print(full_dataset["mean_judgment"].describe())

(10123, 7)
count    10123.000000
mean         2.682042
std          1.087673
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          4.000000
Name: mean_judgment, dtype: float64


In [13]:
full_dataset["judgment_bucket"] = full_dataset["mean_judgment"].round()

sample = (
    full_dataset.groupby("judgment_bucket", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 15), random_state=42))
)

print(sample.shape)
print(sample["judgment_bucket"].value_counts())

(60, 8)
judgment_bucket
1.0    15
2.0    15
3.0    15
4.0    15
Name: count, dtype: int64


/tmp/ipykernel_805/3166299749.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 15), random_state=42))


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import os


In [16]:
os.makedirs('/content/drive/MyDrive/seminar_paper', exist_ok=True)

In [17]:
sample.to_csv('/content/drive/MyDrive/seminar_paper/sample_pairs.csv', index=False)

In [18]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.9 MB/s eta 0:00:00


In [ ]:
from groq import Groq
import os

os.environ["GROQ_API_KEY"] = "ADD YOUR GROQ API KEY HERE"  # paste your actual key
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [20]:
def build_durel_prompt(sentence1, sentence2, word):
    return f"""You are an expert linguistic annotator evaluating semantic change and usage relatedness using the DURel framework.

Task: Rate how related the meaning of the target word "{word}" is between Sentence 1 and Sentence 2 on a 4-point scale:
4 = Identical: exact same meaning in both sentences.
3 = Closely related: meanings are related, differing slightly with context.
2 = Distantly related: meanings are loosely connected (different senses).
1 = Unrelated: meanings share no connection.

Sentence 1: {sentence1}
Sentence 2: {sentence2}

Instructions: Analyze the contextual meanings internally, then provide your final rating as a single integer (1, 2, 3, or 4) on the very last line prefixed with "Rating: "."""

In [44]:
import time
import pandas as pd
from scipy.stats import spearmanr
import re

# --- SAFETY CHECK: Ensure eval_sample exists ---
if 'eval_sample' not in locals() and 'eval_sample' not in globals():
    print("eval_sample not found in memory. Re-creating eval_sample from full_dataset...")
    if 'full_dataset' not in locals():
        # Fallback if full_dataset is also missing: reload a small sample
        import os
        words = os.listdir("DWUG-English/dwug_en/data")[:15]
        all_pairs = []
        for word in words:
            try:
                u = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/uses.csv", sep="\t")
                j = pd.read_csv(f"DWUG-English/dwug_en/data/{word}/judgments.csv", sep="\t")
                j = j[j["judgment"] >= 1]
                id2ctx = u.set_index("identifier")["context"]
                j["sentence1"] = j["identifier1"].map(id2ctx)
                j["sentence2"] = j["identifier2"].map(id2ctx)
                pg = j.groupby(["identifier1","identifier2"]).agg(
                    sentence1=("sentence1","first"),
                    sentence2=("sentence2","first"),
                    mean_judgment=("judgment","mean")
                ).reset_index()
                pg["lemma"] = word
                all_pairs.append(pg)
            except:
                pass
        full_dataset = pd.concat(all_pairs, ignore_index=True)

    full_dataset["judgment_bucket"] = full_dataset["mean_judgment"].round()
    eval_sample = (
        full_dataset.groupby("judgment_bucket", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), 15), random_state=42))
    )

def build_few_shot_durel_prompt(sentence1, sentence2, word):
    return f"""You are an expert linguistic annotator evaluating semantic change and usage relatedness using the DURel framework.

Rate how related the meaning of the target word "{word}" is between Sentence 1 and Sentence 2 on this 4-point scale:
4 = Identical: exact same meaning in both sentences.
3 = Closely related: meanings are related, differing slightly with context.
2 = Distantly related: meanings are loosely connected (different senses).
1 = Unrelated: meanings share no connection.

Here are reference examples:

Example 1 (Score: 4 - Identical):
Word: bank
Sentence 1: He sat on the river bank fishing.
Sentence 2: The muddy bank of the stream was slippery.
Rating: 4

Example 2 (Score: 1 - Unrelated):
Word: bank
Sentence 1: I need to deposit money at the local bank.
Sentence 2: He sat on the river bank fishing.
Rating: 1

Now evaluate this target pair:
Word: {word}
Sentence 1: {sentence1}
Sentence 2: {sentence2}

Analyze the contextual meanings internally, then provide your final rating as a single integer (1, 2, 3, or 4) on the very last line prefixed with "Rating: "."""

# Run few-shot inference on your eval_sample
few_shot_results = []

print("Running Few-Shot Prompting Evaluation...")
for idx, row in eval_sample.reset_index(drop=True).iterrows():
    prompt = build_few_shot_durel_prompt(row["sentence1"], row["sentence2"], row["lemma"])

    score = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=300,
                reasoning_effort="low"
            )
            text = response.choices[0].message.content.strip()

            # Extract rating using regex
            match = re.search(r"Rating:\s*([1-4])", text, re.IGNORECASE)
            if match:
                score = int(match.group(1))
                break
            else:
                fallback_match = re.findall(r"\b([1-4])\b", text)
                if fallback_match:
                    score = int(fallback_match[-1])
                    break
        except Exception as e:
            time.sleep(2)

    few_shot_results.append({
        "lemma": row["lemma"],
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
        "human_judgment": row["mean_judgment"],
        "llm_judgment": score
    })
    time.sleep(1)

# Clean and compute metrics for Few-Shot
fs_df = pd.DataFrame(few_shot_results).dropna(subset=["llm_judgment"])
fs_rho, fs_p_value = spearmanr(fs_df["human_judgment"], fs_df["llm_judgment"])
fs_mae = (fs_df["human_judgment"] - fs_df["llm_judgment"]).abs().mean()

print(f"\n=== FEW-SHOT RESULTS ===")
print(f"Successful predictions: {len(fs_df)} / {len(eval_sample)}")
print(f"Spearman Correlation (rho): {fs_rho:.3f} (p-value: {fs_p_value:.4e})")
print(f"Mean Absolute Error (MAE): {fs_mae:.3f}")

eval_sample not found in memory. Re-creating eval_sample from full_dataset...
Running Few-Shot Prompting Evaluation...


/tmp/ipykernel_805/2902220199.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 15), random_state=42))



=== FEW-SHOT RESULTS ===
Successful predictions: 57 / 60
Spearman Correlation (rho): 0.568 (p-value: 4.1487e-06)
Mean Absolute Error (MAE): 0.705


In [21]:
import re
import time

def get_robust_llm_judgment(sentence1, sentence2, word, max_retries=3):
    prompt = build_durel_prompt(sentence1, sentence2, word)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=300,
                reasoning_effort="low"
            )
            text = response.choices[0].message.content.strip()

            # Try to match explicit "Rating: X" pattern
            match = re.search(r"Rating:\s*([1-4])", text, re.IGNORECASE)
            if match:
                return int(match.group(1))

            # Fallback to last isolated digit 1-4
            fallback_match = re.findall(r"\b([1-4])\b", text)
            if fallback_match:
                return int(fallback_match[-1])

        except Exception as e:
            print(f"API error: {e} — retrying in 3s")
            time.sleep(3)
    return None

In [28]:
test_row = sample.iloc[0]
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": build_durel_prompt(test_row["sentence1"], test_row["sentence2"], test_row["lemma"])}],
    temperature=0,
    max_tokens=600,
    reasoning_effort="low"
)
print(repr(response.choices[0].message.content))
print(response.choices[0].finish_reason)
print(response.usage)

'Rating: 1'
stop
CompletionUsage(completion_tokens=132, prompt_tokens=278, total_tokens=410, completion_time=0.14003664, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=119), prompt_time=0.01516711, prompt_tokens_details=None, queue_time=0.020948467, total_time=0.15520375)


In [25]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
for m in resp.json()["data"]:
    print(m["id"])

openai/gpt-oss-120b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
allam-2-7b
canopylabs/orpheus-arabic-saudi
whisper-large-v3
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m


In [30]:
results = []

for idx, row in sample.iterrows():
    score = get_robust_llm_judgment(row["sentence1"], row["sentence2"], row["lemma"])
    results.append({
        "lemma": row["lemma"],
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
        "human_judgment": row["mean_judgment"],
        "llm_judgment": score
    })
    time.sleep(5)
    if idx % 10 == 0:
        pd.DataFrame(results).to_csv("/content/drive/MyDrive/seminar_paper/groq_results_partial.csv", index=False)
        print(f"Saved checkpoint at row {idx}, last score: {score}")

results_df = pd.DataFrame(results)
results_df.to_csv("/content/drive/MyDrive/seminar_paper/groq_results_final.csv", index=False)
print(results_df.shape)
print(results_df.head(10))
print(f"\nNumber of failed (None) judgments: {results_df['llm_judgment'].isna().sum()}")

Saved checkpoint at row 1270, last score: None
Saved checkpoint at row 6340, last score: 2
Saved checkpoint at row 9410, last score: 3
(60, 5)
       lemma                                          sentence1  \
0     rag_nn  Christopher rearranged the floor plan for maxi...   
1    prop_nn  I have sought that moment out, or it has sough...   
2    prop_nn  Perhaps the prop * tiomi of young men ruined i...   
3     pin_vb   Why, I never heard of it before,  said Miss J...   
4    prop_nn  PROPS: A rambunctious peformer with a snappy t...   
5    prop_nn  The monster prop, which cost $320,000, stretch...   
6  record_nn  When the veteran rapper and actor LL Cool J's ...   
7   twist_nn  Let the graft be carefully held in it &; due p...   
8  record_nn  President Carter, who listens to classical rec...   
9   graft_nn  They wore lace and brocade, leather and rags, ...   

                                           sentence2  human_judgment  \
0  Whatever the rags paid the paparazzi, it was

In [31]:
print(results_df[results_df["llm_judgment"].isna()])

       lemma                                          sentence1  \
2    prop_nn  Perhaps the prop * tiomi of young men ruined i...   
9   graft_nn  They wore lace and brocade, leather and rags, ...   
55   prop_nn  wants, indeed, Mrs. Haller may have relieved; ...   

                                            sentence2  human_judgment  \
2   My eldest Boy properly and successfully settle...             1.0   
9   A graft taken from a tree enfeebled by disease...             1.0   
55   I have not mentioned them; but they are main ...             4.0   

    llm_judgment  
2            NaN  
9            NaN  
55           NaN  


In [32]:
clean_results = results_df.dropna(subset=["llm_judgment"]).copy()
print(clean_results.shape)  # should be (58, 5)

(57, 5)


In [33]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(clean_results["human_judgment"], clean_results["llm_judgment"])
print(f"Spearman correlation: {rho:.3f}")
print(f"P-value: {p_value:.4f}")

Spearman correlation: 0.584
P-value: 0.0000


In [34]:
from scipy.stats import spearmanr

# Compute Mean Absolute Error (MAE)
clean_results["error"] = (clean_results["llm_judgment"] - clean_results["human_judgment"]).abs()
mae = clean_results["error"].mean()
print(f"Mean Absolute Error (MAE): {mae:.3f}")

# Per-lemma correlation check to see where LLMs perform best/worst
lemma_stats = []
for lemma, group in clean_results.groupby("lemma"):
    if len(group) > 3:
        r, _ = spearmanr(group["human_judgment"], group["llm_judgment"])
        lemma_stats.append({"lemma": lemma, "spearman_rho": r, "count": len(group)})

lemma_df = pd.DataFrame(lemma_stats).sort_values(by="spearman_rho", ascending=False)
print("\n--- Per-Lemma Performance Summary ---")
print(lemma_df.to_string(index=False))

Mean Absolute Error (MAE): 0.629

--- Per-Lemma Performance Summary ---
       lemma  spearman_rho  count
    graft_nn      1.000000      4
     lane_nn      0.942809      4
      rag_nn      0.888889      4
    twist_nn      0.740436      5
   record_nn      0.616667      6
      pin_vb      0.486664      5
     prop_nn      0.296174      5
     part_nn      0.047036     13
multitude_nn           NaN      4


/tmp/ipykernel_805/3225428310.py:12: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = spearmanr(group["human_judgment"], group["llm_judgment"])


In [35]:
# Extract top disagreements for qualitative analysis in your paper
worst_cases = clean_results.sort_values(by="error", ascending=False).head(5)

print("=== TOP DIVERGENCE CASES (FOR QUALITATIVE ANALYSIS SECTION) ===")
for idx, row in worst_cases.iterrows():
    print(f"Lemma: {row['lemma']} | Human: {row['human_judgment']:.2f} | LLM: {row['llm_judgment']} | Absolute Error: {row['error']:.2f}")
    print(f"  [S1]: {row['sentence1']}")
    print(f"  [S2]: {row['sentence2']}")
    print("-" * 60)

=== TOP DIVERGENCE CASES (FOR QUALITATIVE ANALYSIS SECTION) ===
Lemma: prop_nn | Human: 4.00 | LLM: 1.0 | Absolute Error: 3.00
  [S1]: The French themselves, advancing from-self-defence to-conquest, struck another prop away.
  [S2]: In addition to the new IL-96, Ilyushin now makes the IL-114, a regional two-engine prop, the four-engine IL-76 cargo jet, and the IL-103, a small, single-engine private plane that in December was actually the first Russian airplane to receive F.A.A.
------------------------------------------------------------
Lemma: part_nn | Human: 3.00 | LLM: 1.0 | Absolute Error: 2.00
  [S1]: The old part was built on and from solid rock, being really a jutting out of a lime stone cliff which had stood high and dry before the water had been dammed up by the heavy timber cribs cutting across the original stream.
  [S2]: Port Smith was rife with citizens ' committees, study groups, and action teams composed of people who had not taken part in civic affairs before.
--------

In [36]:
clean_results["diff"] = clean_results["llm_judgment"] - clean_results["human_judgment"]
print(clean_results["diff"].describe())

count    57.000000
mean     -0.184211
std       0.948414
min      -3.000000
25%      -1.000000
50%       0.000000
75%       0.000000
max       2.000000
Name: diff, dtype: float64


In [37]:
worst_disagreements = clean_results.reindex(clean_results["diff"].abs().sort_values(ascending=False).index)
print(worst_disagreements[["lemma", "sentence1", "sentence2", "human_judgment", "llm_judgment", "diff"]].head(10))

        lemma                                          sentence1  \
52    prop_nn  The French themselves, advancing from-self-def...   
38    part_nn  The old part was built on and from solid rock,...   
44  record_nn  // SAILING -- PHILIPPE JEANTOT, in his 60-foot...   
39     pin_vb  He couldn't pin down a definite number, but he...   
53    part_nn  The qualified negative of the president, diffe...   
18   thump_nn  I left the notebook beside the Castles on the ...   
22    part_nn  Rome, Florence, Venice, Milan, Turin, every pa...   
21   grain_nn  The continual repairs of temporary dams, which...   
20    part_nn  There is a set of men, in all parts of the cou...   
26   twist_nn  And so I thought when I got near enough for my...   

                                            sentence2  human_judgment  \
52  In addition to the new IL-96, Ilyushin now mak...        4.000000   
38  Port Smith was rife with citizens ' committees...        3.000000   
44  Suffice it, that the villain

In [38]:
clean_results["human_bucket"] = clean_results["human_judgment"].round()
print(clean_results.groupby("human_bucket")["llm_judgment"].mean())

human_bucket
1.0    1.538462
2.0    2.133333
3.0    2.466667
4.0    3.285714
Name: llm_judgment, dtype: float64


In [40]:
# check how many sentences in your full sample contain suspicious patterns (digits:digits)
import re
suspicious = clean_results[
    clean_results["sentence1"].str.contains(r'\d+:\d+', regex=True) |
    clean_results["sentence2"].str.contains(r'\d+:\d+', regex=True)
]
print(suspicious.shape)
print(suspicious[["lemma","sentence1","sentence2"]])

(0, 8)
Empty DataFrame
Columns: [lemma, sentence1, sentence2]
Index: []


In [42]:
# find this specific pair in your original full_dataset (before sampling), which has n_annotators
check = full_dataset[
    (full_dataset["sentence1"].str.contains("struck another prop away", regex=False)) |
    (full_dataset["sentence2"].str.contains("struck another prop away", regex=False))
]
print(check[["sentence1","sentence2","mean_judgment","n_annotators"]])

                                                                                                                                                                                                                                                                                                    sentence1  \
80    T was evening; and the lamp, Set near, shone full upon her placid face, Its snowy white illuming, while they stood Gazing as on her loveliness in sleep, The enfeebled mother on the father's arm Heavily hanging, like the slender flower On its firm prop, when loaded down with rain Or morning dew.   
175                                                                                                                                                                                                                                                  and props of the undertaking -- but few tributes to pay.   
183                                                                                  

In [45]:
# Install sentence-transformers if not already available
!pip install sentence-transformers -q

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from scipy.stats import spearmanr

# 1. Load a lightweight, high-performing free embedding model
print("Loading sentence embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Compute cosine similarity for each pair in your eval_sample
baseline_results = []

print("Computing embedding baseline similarities...")
for idx, row in eval_sample.reset_index(drop=True).iterrows():
    # Encode both sentences
    emb1 = model.encode(row["sentence1"], convert_to_tensor=True)
    emb2 = model.encode(row["sentence2"], convert_to_tensor=True)

    # Calculate cosine similarity (range: 0 to 1)
    cos_sim = util.cos_sim(emb1, emb2).item()

    # Scale cosine similarity (0 to 1) to match DURel scale (1 to 4) roughly for comparison,
    # or keep raw similarity to test monotonic correlation via Spearman rho.
    # Spearman rank correlation doesn't care about linear scaling, only ranking order!

    baseline_results.append({
        "lemma": row["lemma"],
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
        "human_judgment": row["mean_judgment"],
        "embedding_similarity": cos_sim
    })

# 3. Evaluate baseline correlation
base_df = pd.DataFrame(baseline_results)
emb_rho, emb_p_value = spearmanr(base_df["human_judgment"], base_df["embedding_similarity"])

print(f"\n=== EMBEDDING BASELINE RESULTS (all-MiniLM-L6-v2) ===")
print(f"Spearman Correlation (rho): {emb_rho:.3f} (p-value: {emb_p_value:.4e})")

Loading sentence embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing embedding baseline similarities...

=== EMBEDDING BASELINE RESULTS (all-MiniLM-L6-v2) ===
Spearman Correlation (rho): 0.200 (p-value: 1.2585e-01)
